<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.3-firestore-vector/notebooks/GCP_Capstone_2.3_Firestore_Vector.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.3 Firestore Vector Search — Your First Vector Database
**Netsetos GenAI Engineering — GCP Capstone**

Store embeddings with Vector(), query with find_nearest(), build a complete RAG pipeline.


## Setup


In [ ]:
!pip install -q google-cloud-firestore google-genai
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
FIRESTORE_LOCATION = 'asia-south1'  # region for the (default) Firestore DB - PERMANENT once created (us-central1 = course default)

# Firestore needs a (default) database in Native mode; enabling the API is not enough.
# Create it once if missing (idempotent). The DB location cannot be changed later.
import subprocess
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print(f'Creating Firestore (default) database in {FIRESTORE_LOCATION} (one-time, ~30s)...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=' + FIRESTORE_LOCATION, '--project', PROJECT_ID], check=False)
else:
    print('Firestore (default) database ready.')

from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google import genai
from google.genai import types

db = firestore.Client(project=PROJECT_ID)
ai = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings: regional-only
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # generation: global endpoint (Gemini 3.x)
collection = db.collection('knowledge_base')
print('Clients initialized')


## Cell 1: Create Vector Indexes

Firestore vector search needs composite indexes. **Run the next cell** to create both (works in Colab, not only Cloud Shell):
- `embedding` — unfiltered `find_nearest()` (Cell 4)
- `category` + `embedding` — **filtered** search by category (Cells 5-7)

Each index takes **~2-5 minutes to build**. If a `find_nearest()` cell raises `FAILED_PRECONDITION: Missing vector index`, the index is still building (or a filtered query needs its own composite index) — wait a bit and re-run.

In [ ]:
# Firestore vector search needs composite VECTOR INDEXES. This lesson uses two:
#   1. embedding                  -> unfiltered find_nearest (Cell 4)
#   2. category (ASC) + embedding -> FILTERED find_nearest by category (Cells 5-7)
# gcloud works in Colab; idempotent (ignores "already exists"). Each index takes
# ~2-5 min to BUILD before the matching query works.
import subprocess
_BASE = (f"gcloud firestore indexes composite create --project={PROJECT_ID} "
         "--collection-group=knowledge_base --query-scope=COLLECTION ")
_VEC = "--field-config=vector-config='{\"dimension\":\"768\",\"flat\":\"{}\"}',field-path=embedding"
_INDEXES = {
    'embedding (unfiltered)': _VEC,
    'category + embedding (filtered)': "--field-config=order=ASCENDING,field-path=category " + _VEC,
}
for _name, _fields in _INDEXES.items():
    _r = subprocess.run(_BASE + _fields, shell=True, capture_output=True, text=True)
    _out = (_r.stdout + _r.stderr).strip()
    if _r.returncode == 0:
        print(f'[{_name}] creation started.')
    elif 'already exists' in _out.lower():
        print(f'[{_name}] already exists.')
    else:
        print(f'[{_name}] {_out}')
# Wait until the indexes finish building - find_nearest() raises FAILED_PRECONDITION
# ("Missing vector index") until every index is READY, not just created.
import time
print('Waiting for indexes to build (~2-5 min; find_nearest fails until READY)...')
for _ in range(40):  # up to ~10 min
    _states = subprocess.run(
        f"gcloud firestore indexes composite list --project={PROJECT_ID} --format='value(state)'",
        shell=True, capture_output=True, text=True).stdout.split()
    if _states and 'CREATING' not in _states:
        print(f'All {len(_states)} indexes READY - find_nearest cells (4-7) will work now.'); break
    time.sleep(15)
else:
    print('Still building after ~10 min. Check: '
          f'!gcloud firestore indexes composite list --project={PROJECT_ID}')

## Cell 2: Store a Single Document with Embedding


In [ ]:
text = 'The transformer architecture uses self-attention mechanisms.'
resp = ai.models.embed_content(
    model='gemini-embedding-001', contents=text,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768))

collection.document('doc_001').set({
    'content': text,
    'embedding': Vector(resp.embeddings[0].values),
    'category': 'ai_ml',
})
print(f'Stored doc_001 with {len(resp.embeddings[0].values)}-dim embedding')


## Cell 3: Batch Ingest 20 Documents


In [ ]:
corpus = [
    ('ai_ml', 'Neural networks learn patterns from labeled training data.'),
    ('ai_ml', 'Transformers use self-attention to process sequences in parallel.'),
    ('ai_ml', 'RAG combines document retrieval with language model generation.'),
    ('ai_ml', 'Fine-tuning adapts a pre-trained model to a specific domain.'),
    ('ai_ml', 'Embeddings convert text into dense numerical vectors.'),
    ('gcp', 'Cloud Run deploys containerized applications serverlessly.'),
    ('gcp', 'BigQuery processes petabytes of data using SQL queries.'),
    ('gcp', 'Firestore is a NoSQL document database with real-time sync.'),
    ('gcp', 'Vertex AI provides managed infrastructure for ML workflows.'),
    ('gcp', 'Cloud Storage offers durable object storage at low cost.'),
    ('india', 'Hyderabad is the capital of Telangana and a major tech hub.'),
    ('india', 'India Digital Public Infrastructure includes UPI and Aadhaar.'),
    ('india', 'DPDP Act 2023 governs personal data protection in India.'),
    ('india', 'IndiaAI Mission allocates 10,000 GPUs for AI research.'),
    ('india', 'Bangalore, Hyderabad, and Pune are top Indian tech cities.'),
    ('python', 'Python list comprehensions provide concise filtering syntax.'),
    ('python', 'Asyncio enables concurrent I/O-bound operations in Python.'),
    ('python', 'Type hints improve code readability and enable analysis.'),
    ('python', 'Virtual environments isolate project dependencies.'),
    ('python', 'Pydantic validates data using Python type annotations.'),
]

texts = [t for _, t in corpus]
embeddings = [ai.models.embed_content(
    model='gemini-embedding-001', contents=t,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)).embeddings[0].values
    for t in texts]

batch = db.batch()
for i, ((cat, text), vec) in enumerate(zip(corpus, embeddings)):
    ref = collection.document(f'doc_{i:03d}')
    batch.set(ref, {'content': text, 'embedding': Vector(vec), 'category': cat})
batch.commit()
print(f'Loaded {len(corpus)} documents')


## Cell 4: find_nearest() — Vector Search


In [ ]:
query_text = 'How does attention work in deep learning?'
q_resp = ai.models.embed_content(
    model='gemini-embedding-001', contents=query_text,
    config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768))

results = collection.find_nearest(
    vector_field='embedding',
    query_vector=Vector(q_resp.embeddings[0].values),
    distance_measure=DistanceMeasure.COSINE,
    limit=5,
    distance_result_field='vector_distance',
).get()

print(f'Query: {query_text}')
for doc in results:
    d = doc.to_dict()
    sim = 1 - d['vector_distance']
    print(f'  [{sim:.4f}] {d.get("category","")} | {d["content"]}')


## Cell 5: Filtered Vector Search


In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter

# Search within 'ai_ml' category only
# NOTE: Requires composite index (category + embedding)
results = collection.where(filter=FieldFilter('category', '==', 'ai_ml')).find_nearest(
    vector_field='embedding',
    query_vector=Vector(q_resp.embeddings[0].values),
    distance_measure=DistanceMeasure.COSINE,
    limit=3,
    distance_result_field='vector_distance',
).get()

print('Filtered search (ai_ml only):')
for doc in results:
    d = doc.to_dict()
    print(f'  [{1-d["vector_distance"]:.4f}] {d["content"]}')


## Cell 6: Complete RAG Pipeline


In [ ]:
def rag_query(question, category=None, top_k=3):
    # Embed query
    q_emb = ai.models.embed_content(
        model='gemini-embedding-001', contents=question,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY', output_dimensionality=768)
    ).embeddings[0].values
    
    # Search
    ref = collection
    if category:
        ref = ref.where(filter=FieldFilter('category', '==', category))
    docs = ref.find_nearest(
        vector_field='embedding', query_vector=Vector(q_emb),
        distance_measure=DistanceMeasure.COSINE,
        limit=top_k, distance_result_field='dist',
        distance_threshold=0.5,
    ).get()
    
    context = '\n'.join([d.to_dict()['content'] for d in docs])
    
    # Generate
    response = gen_client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Context:\n{context}\n\nQuestion: {question}',
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    
    print(f'Query: {question}')
    print(f'Sources: {len(docs)} documents')
    for d in docs:
        dd = d.to_dict()
        print(f'  [{1-dd["dist"]:.4f}] {dd["content"][:60]}...')
    print(f'\nAnswer: {response.text}')

rag_query('How do transformers process sequences?')
print('\n' + '='*60 + '\n')
rag_query('What is India\'s data protection law?', category='india')


## Cell 7: FirestoreRAG Production Module


In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter

class FirestoreRAG:
    def __init__(self, project, collection_name='knowledge_base'):
        self.db = firestore.Client(project=project)
        self.ai = genai.Client(enterprise=True, project=project, location='us-central1')
        self.col = self.db.collection(collection_name)
    
    def embed(self, text, task='RETRIEVAL_DOCUMENT'):
        return self.ai.models.embed_content(
            model='gemini-embedding-001', contents=text,
            config={'task_type': task, 'output_dimensionality': 768}
        ).embeddings[0].values
    
    def search(self, query, category=None, top_k=5):
        q_vec = self.embed(query, task='RETRIEVAL_QUERY')
        ref = self.col
        if category: ref = ref.where(filter=FieldFilter('category', '==', category))
        docs = ref.find_nearest(
            vector_field='embedding', query_vector=Vector(q_vec),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k, distance_result_field='dist'
        ).get()
        return [{'content': d.to_dict()['content'],
                 'similarity': 1 - d.to_dict()['dist']} for d in docs]

# Test
rag = FirestoreRAG(PROJECT_ID)
results = rag.search('What is RAG?')
for r in results:
    print(f'  [{r["similarity"]:.4f}] {r["content"]}')


## ✅ Lesson 2.3 Complete!

- ✅ Created flat vector index via gcloud
- ✅ Stored documents with Vector() embeddings
- ✅ Queried with find_nearest() + COSINE distance
- ✅ Converted cosine distance → similarity
- ✅ Filtered vector search with .where()
- ✅ distance_threshold for quality filtering
- ✅ Full RAG pipeline: embed → search → generate
- ✅ FirestoreRAG production module

**Next: Lesson 2.4 — AlloyDB pgvector & BigQuery Vector Search**
